# Phase 4: Machine Learning Development

## Objective

The objective of this phase is to develop, evaluate, and optimize machine learning models for predicting loan risk. The processed dataset from Phase 3 is used to train multiple classification algorithms, compare their performance, and select the best-performing model for deployment.

## Workflow

* Load the processed dataset
* Perform train-test split
* Train baseline model
* Evaluate model performance
* Train multiple machine learning models
* Compare model performance
* Perform cross-validation
* Tune hyperparameters
* Analyze feature importance
* Save the best-performing model


In [48]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import(
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    RandomizedSearchCV
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.pipeline import Pipeline

import joblib

from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)


In [63]:
df = pd.read_csv("../data/processed/loan_data_processed.csv")

In [65]:
df.shape

(2260668, 90)

In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2260668 entries, 0 to 2260667
Data columns (total 90 columns):
 #   Column                          Dtype  
---  ------                          -----  
 0   loan_amnt                       int64  
 1   funded_amnt                     int64  
 2   funded_amnt_inv                 float64
 3   term                            int64  
 4   int_rate                        float64
 5   installment                     float64
 6   grade                           int64  
 7   sub_grade                       int64  
 8   home_ownership                  int64  
 9   annual_inc                      float64
 10  verification_status             int64  
 11  loan_status                     int64  
 12  pymnt_plan                      int64  
 13  purpose                         int64  
 14  addr_state                      int64  
 15  dti                             float64
 16  delinq_2yrs                     float64
 17  inq_last_6mths                  float6

In [68]:
df['loan_status'].value_counts()

loan_status
5    1041952
1     919695
0     261655
8      21897
6       8952
7       3737
4       1988
3        761
2         31
Name: count, dtype: int64

In [70]:
X = df.drop('loan_status', axis=1)
y = df['loan_status']

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [72]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent',random_state=42)

dummy.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'most_frequent'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](9,)","[0.12,0.41,0. ,...,0. ,0. ,0.01]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](9,)","[0,1,2,...,6,7,8]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X` hasfeature names that are all strings.","ndarray[object](89,)","['loan_amnt','funded_amnt','funded_amnt_inv',...,'loan_to_income_ratio', 'installment_to_income_ratio','emp_length_numeric']"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,9
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,89
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [73]:
y_pred = dummy.predict(X_test)
y_prob = dummy.predict_proba(X_test)[:, 1]

In [76]:
baseline_metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, y_prob),
}

baseline_metrics

ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [77]:
print(df['loan_status'].value_counts())

loan_status
5    1041952
1     919695
0     261655
8      21897
6       8952
7       3737
4       1988
3        761
2         31
Name: count, dtype: int64


In [79]:
print(df["loan_status"].unique())

[1 5 8 6 0 7 2 4 3]


In [80]:
print(y_train.unique())
print(y_train.value_counts())

[5 0 1 6 7 8 4 3 2]
loan_status
5    833561
1    735756
0    209324
8     17517
6      7162
7      2990
4      1590
3       609
2        25
Name: count, dtype: int64


In [81]:
label_encoders["loan_status"].classes_

array(['Charged Off', 'Current', 'Default',
       'Does not meet the credit policy. Status:Charged Off',
       'Does not meet the credit policy. Status:Fully Paid', 'Fully Paid',
       'In Grace Period', 'Late (16-30 days)', 'Late (31-120 days)'],
      dtype=object)